# Reward Model Training for Arabic Translation

This notebook trains a reward model for RLHF-based Arabic translation using synthetic preference data.

**Configuration:**
- Supports both GPU and CPU execution
- Optimized batch sizes for 16GB GPU memory
- Configurable for local or cloud environments

## Key Features and Updates

### Model Training Improvements:

1. **Gradient Optimization**: Explicit unfreezing logic ensures that unfrozen layers and the reward head receive proper gradient updates during training.

2. **Debug Mode**: Toggle `DEBUG_MODE = True` in the configuration cell to train on 2,000 samples for quick iteration (approximately 15-20 minutes).

3. **Accuracy Monitoring**: The training loop tracks the **Reward Margin** (chosen - rejected) in real-time to monitor learning progress.

4. **Performance Optimization**: 
   - `RM_MAX_LENGTH = 512` for handling longer sentences
   - `RM_GRADIENT_ACCUMULATION_STEPS = 12` for training stability
   - `RM_UNFROZEN_LAYERS = 8` for improved language understanding

### Training Metrics:

- **Accuracy (acc)**: Expected to increase from approximately 0.50 to above 0.60 with successful learning
- **Margin**: Should show steady increase (e.g., from 0.01 to 0.15+) as the model learns
- **Loss**: Should decrease consistently throughout training

### Usage Modes:

1. **Quick Test**: Set `DEBUG_MODE = True` for 15-20 minute training run
2. **Full Training**: Set `DEBUG_MODE = False` for complete dataset training

## Install Required Packages

Install all necessary dependencies for training the reward model.

In [ ]:
!pip install -q transformers accelerate torch datasets trl peft bitsandbytes
!pip install -q wandb tqdm
print("✅ Packages installed!")

## Configuration and Setup

Configure the training environment, set hyperparameters, and initialize the training pipeline.

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass

# ===========================
# ENVIRONMENT SETUP
# ===========================

# Set up local paths
PROJECT_DIR = Path.cwd()
OUTPUTS_DIR = PROJECT_DIR / "outputs"
MODELS_DIR = PROJECT_DIR / "models"

print("Environment Configuration")
print(f"  Project Directory: {PROJECT_DIR}")
print(f"  Data Directory: {OUTPUTS_DIR}")
print(f"  Models Directory: {MODELS_DIR}")

# Create directories
MODELS_DIR.mkdir(exist_ok=True, parents=True)
REWARD_MODEL_COLD_START = MODELS_DIR / "reward_model_coldstart"

# ===========================
# GPU CONFIGURATION
# ===========================

print(f"\n{'='*60}")
print("GPU Configuration")
print(f"{'='*60}")

if torch.cuda.is_available():
    NUM_GPUS = torch.cuda.device_count()
    print(f"✅ GPU Available: {NUM_GPUS} GPU(s)")
    for i in range(NUM_GPUS):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"   GPU {i}: {gpu_name} ({gpu_memory:.1f} GB)")
else:
    NUM_GPUS = 0
    print("⚠️  No GPU detected - training will be very slow!")

# ===========================
# DEBUG MODE (Fast-Test Toggle)
# ===========================
DEBUG_MODE = True  # Set to False for full training
print(f"\n{'='*60}")
if DEBUG_MODE:
    print("⚠️  DEBUG MODE ENABLED - Training on subset for fast testing")
else:
    print("✅ FULL TRAINING MODE")
print(f"{'='*60}\n")

# ===========================
# HYPERPARAMETERS (Kaggle Optimized)
# ===========================

# Model Configuration
REWARD_BASE_MODEL = "google/gemma-2-2b"  # Smaller model for Kaggle GPU

# Training Configuration (Optimized for single T4 GPU - 16GB)
# Note: Using single GPU to avoid DataParallel memory overhead
RM_LEARNING_RATE = 1e-4  # Higher LR for training new reward head from scratch
RM_BATCH_SIZE = 4  # Conservative for single T4 GPU
RM_EPOCHS = 1
RM_MAX_LENGTH = 512  # Increased for long sentences
RM_GRADIENT_ACCUMULATION_STEPS = 4  # Reduced for faster updates (was 12)
# HYPERPARAMETERS
# Model Architecture
RM_HEAD_TYPE = "mlp"
RM_HIDDEN_DIM = 256
REWARD_BASE_MODEL = "google/gemma-2-2b"
RM_DROPOUT = 0.15
# Training Configuration (Optimized for 16GB GPU)
RM_LEARNING_RATE = 1e-4  # Higher LR for training new reward head from scratch
RM_BATCH_SIZE = 4  # Conservative for 16GB GPU
RM_EPOCHS = 1
RM_MAX_LENGTH = 512  # For handling long sentences
RM_GRADIENT_ACCUMULATION_STEPS = 4  # For stable training

# Model Architecture
RM_HEAD_TYPE = "mlp"
RM_HIDDEN_DIM = 256
RM_UNFROZEN_LAYERS = 12  # More layers for complex Arabic translation nuances
RM_DROPOUT = 0.15
RM_WEIGHT_DECAY = 0.01

# Flash Attention
USE_FLASH_ATTENTION = False
ATTN_IMPLEMENTATION = "eager"

# Weights & Biases (optional)
USE_WANDB = False  # Set to True if you want to track experiments
WANDB_PROJECT = "rlhf-arabic-translation"

# Random Seed
SEED = 42

print("Training Configuration")
print(f"{'='*60}")
print(f"Batch Size: {RM_BATCH_SIZE}")
print(f"Effective Batch: {RM_BATCH_SIZE * RM_GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning Rate: {RM_LEARNING_RATE}")
print(f"Epochs: {RM_EPOCHS}")
print(f"Max Length: {RM_MAX_LENGTH}")
print(f"Unfrozen Layers: {RM_UNFROZEN_LAYERS}")
print(f"Gradient Accumulation: {RM_GRADIENT_ACCUMULATION_STEPS}")
print(f"GPU Strategy: {'DataParallel (multi-GPU)' if NUM_GPUS > 1 else 'Single GPU'}")
print(f"{'='*60}\n")

# ===========================
# UTILITY FUNCTIONS
# ===========================


### Required Input Data

Ensure your preference data files are located in the `outputs` directory relative to this notebook:
- `en-ar-preferences.jsonl` (English to Arabic preferences)
- `fr-ar-preferences.jsonl` (French to Arabic preferences)

**Expected JSONL Format:**
```json
{
  "source": "source text",
  "chosen": "better translation",
  "rejected": "worse translation",
  "chosen_score": 0.85,
  "rejected_score": 0.65,
  "margin": 0.20,
  "source_lang": "en"
}
```

In [ ]:
# Already imported in configuration cell above
# Just print training info
print("Reward Model Training Configuration")
print(f"   Batch size: {RM_BATCH_SIZE}")
print(f"   Learning rate: {RM_LEARNING_RATE}")
print(f"   Epochs: {RM_EPOCHS}")
print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Load Synthetic Preference Data

Load and validate the preference pairs from the English and French translation datasets.

In [ ]:
# Load English and French preference files
print("Loading synthetic preference data...")
print(f"  Looking in: {OUTPUTS_DIR}")

preference_data = []

# Load English preferences
en_pref_file = OUTPUTS_DIR / "en-ar-preferences.jsonl"
print(f"  EN file: {en_pref_file}")

if en_pref_file.exists():
    with open(en_pref_file, 'r', encoding='utf-8') as f:
        en_count = 0
        for line in f:
            preference_data.append(json.loads(line))
            en_count += 1
    print(f"✓ Loaded {en_count} English preference pairs")
else:
    print(f"⚠️  EN preferences file not found: {en_pref_file}")

# Load French preferences
fr_pref_file = OUTPUTS_DIR / "fr-ar-preferences.jsonl"
print(f"  FR file: {fr_pref_file}")

if fr_pref_file.exists():
    with open(fr_pref_file, 'r', encoding='utf-8') as f:
        fr_count = 0
        for line in f:
            preference_data.append(json.loads(line))
            fr_count += 1
    print(f"✓ Loaded {fr_count} French preference pairs")
else:
    print(f"⚠️  FR preferences file not found: {fr_pref_file}")

print(f"\nTotal preference pairs (EN + FR): {len(preference_data)}")

if len(preference_data) == 0:
    raise ValueError("No preference data loaded! Check file paths in the outputs directory.")

# ===========================
# HARD FILTER: Remove weak preferences (margin <= 0.05)
# ===========================
original_count = len(preference_data)
preference_data = [item for item in preference_data if item['margin'] > 0.05]
filtered_count = original_count - len(preference_data)
print(f"\n🔍 HARD FILTER APPLIED:")
print(f"   Removed {filtered_count} samples with margin ≤ 0.05")
print(f"   Kept {len(preference_data)} samples with strong preferences (margin > 0.05)")
print(f"   Filter rate: {100*filtered_count/original_count:.1f}%")

if len(preference_data) == 0:
    raise ValueError("No data remaining after filtering! All samples had margin ≤ 0.05.")

# ===========================
# DEBUG MODE SUBSAMPLING
# ===========================
if DEBUG_MODE:
    print(f"\n⚠️  DEBUG MODE: Subsampling 2000 pairs for fast testing...")
    random.shuffle(preference_data)
    preference_data = preference_data[:2000]
    print(f"   Reduced dataset to {len(preference_data)} samples")

# Split into train/validation (90/10 split)
random.shuffle(preference_data)
train_size = int(0.9 * len(preference_data))
train_data = preference_data[:train_size]
val_data = preference_data[train_size:]

print(f"\nTrain: {len(train_data)} pairs")
print(f"Validation: {len(val_data)} pairs")

# Language breakdown
en_train = sum(1 for item in train_data if item.get('source_lang') == 'en')
fr_train = sum(1 for item in train_data if item.get('source_lang') == 'fr')
en_val = sum(1 for item in val_data if item.get('source_lang') == 'en')
fr_val = sum(1 for item in val_data if item.get('source_lang') == 'fr')

print(f"\nLanguage breakdown (train):")
print(f"  English: {en_train} ({100*en_train/len(train_data):.1f}%)")
print(f"  French: {fr_train} ({100*fr_train/len(train_data):.1f}%)")
print(f"\nLanguage breakdown (validation):")
print(f"  English: {en_val} ({100*en_val/len(val_data):.1f}%)")
print(f"  French: {fr_val} ({100*fr_val/len(val_data):.1f}%)")

In [ ]:
# Validate preference data quality
print("\nValidating preference data quality...")
sample_margins = [item['margin'] for item in preference_data]
print(f"Margin statistics (all {len(sample_margins)} samples):")
print(f"  Min: {min(sample_margins):.4f}")
print(f"  Max: {max(sample_margins):.4f}")
mean_margin = sum(sample_margins)/len(sample_margins)
print(f"  Mean: {mean_margin:.4f}")
if len(sample_margins) > 1:
    std_margin = (sum((x - mean_margin)**2 for x in sample_margins)/len(sample_margins))**0.5
    print(f"  Std: {std_margin:.4f}")

# Check for degenerate cases
zero_margin_count = sum(1 for item in preference_data if item['margin'] < 0.01)
print(f"  Samples with margin < 0.01: {zero_margin_count}/{len(preference_data)}")
if zero_margin_count > len(preference_data) * 0.5:
    print("  ⚠️  WARNING: >50% of samples have near-zero margins (weak preference signal)")

# Quality by language
if any('source_lang' in item for item in preference_data):
    en_margins = [item['margin'] for item in preference_data if item.get('source_lang') == 'en']
    fr_margins = [item['margin'] for item in preference_data if item.get('source_lang') == 'fr']
    
    if en_margins:
        en_mean = sum(en_margins)/len(en_margins)
        print(f"\nEnglish margin stats (n={len(en_margins)}): mean={en_mean:.4f}")
    if fr_margins:
        fr_mean = sum(fr_margins)/len(fr_margins)
        print(f"French margin stats (n={len(fr_margins)}): mean={fr_mean:.4f}")

## Preference Dataset Class

Define the PyTorch dataset class for handling pairwise preference data during training.

In [ ]:
class PreferenceDataset(Dataset):
    """Dataset for pairwise preference data"""
    
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format: "Source: ... \nTranslation: ..."
        chosen_text = f"Source: {item['source']}\nTranslation: {item['chosen']}"
        rejected_text = f"Source: {item['source']}\nTranslation: {item['rejected']}"
        
        # Tokenize
        chosen_tokens = self.tokenizer(
            chosen_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        rejected_tokens = self.tokenizer(
            rejected_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'chosen_input_ids': chosen_tokens['input_ids'].squeeze(0),
            'chosen_attention_mask': chosen_tokens['attention_mask'].squeeze(0),
            'rejected_input_ids': rejected_tokens['input_ids'].squeeze(0),
            'rejected_attention_mask': rejected_tokens['attention_mask'].squeeze(0),
            'margin': item['margin']  # For analysis
        }

print("PreferenceDataset class defined (max_length=512)")

## Reward Model Architecture

Define the reward model architecture combining a base language model with a custom reward head.

In [ ]:
class RewardModel(nn.Module):
    """Reward model with base LM + reward head"""
    
    def __init__(self, base_model, hidden_dim=256, head_type='mlp', dropout=0.15):
        super().__init__()
        self.base_model = base_model
        self.head_type = head_type
        
        # Get hidden size from base model
        self.hidden_size = base_model.config.hidden_size
        
        # Reward head
        if head_type == 'linear':
            self.reward_head = nn.Linear(self.hidden_size, 1)
        elif head_type == 'mlp':
            self.reward_head = nn.Sequential(
                nn.Linear(self.hidden_size, hidden_dim),
                nn.LayerNorm(hidden_dim),  # Add LayerNorm for stability
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            )
        else:
            raise ValueError(f"Unknown head_type: {head_type}")
        
        # Better initialization: Xavier for hidden, small for output
        with torch.no_grad():
            if head_type == 'mlp':
                # Xavier initialization for hidden layer
                nn.init.xavier_normal_(self.reward_head[0].weight)
                nn.init.zeros_(self.reward_head[0].bias)
                # Small random init for output layer
                nn.init.normal_(self.reward_head[-1].weight, mean=0, std=0.01)
                nn.init.zeros_(self.reward_head[-1].bias)
    
    def forward(self, input_ids, attention_mask):
        # Get base model outputs with hidden states
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,  # Need hidden states for reward computation
            use_cache=False  # Disable cache to avoid device placement issues with multi-GPU
        )
        
        # Get last hidden state from hidden_states tuple
        # hidden_states is a tuple of all layer outputs, last one is what we need
        hidden_states = outputs.hidden_states[-1]  # [batch, seq_len, hidden_size]
        
        # Pool: use last token representation (similar to value head in PPO)
        # Get the last non-padding token for each sequence
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = hidden_states.shape[0]
        pooled = hidden_states[torch.arange(batch_size), sequence_lengths]
        
        # Apply reward head
        reward = self.reward_head(pooled)  # [batch, 1]
        
        return reward.squeeze(-1)  # [batch]

print("RewardModel class defined")

## Load Base Model and Initialize Reward Model

Load the pre-trained language model and initialize the reward model components.

## HuggingFace Authentication

**Required for Gated Models (e.g., Gemma)**

To access gated models:
1. Accept model terms at: https://huggingface.co/google/gemma-2-2b
2. Create access token at: https://huggingface.co/settings/tokens
3. Configure authentication using one of the methods below:
   - Set environment variable: `HF_TOKEN` or `HUGGINGFACE_TOKEN`
   - Run `huggingface-cli login` in terminal
   - Manually call `login(token="your_token_here")` in the cell below

In [ ]:
from huggingface_hub import login
import os

print("=" * 80)
print("HUGGINGFACE AUTHENTICATION")
print("=" * 80)

# Method 1: Environment variable
if "HF_TOKEN" in os.environ or "HUGGINGFACE_TOKEN" in os.environ:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    login(token=token)
    print("Authenticated using environment variable")
else:
    # Method 2: Cached token
    try:
        login()
        print("Authenticated using cached token")
    except:
        print("No HuggingFace token found!")
        print("\nAuthentication options:")
        print("  1. Run: huggingface-cli login")
        print("  2. Set HF_TOKEN environment variable")
        print("  3. Uncomment and use: login(token='your_token_here')")
        
print("=" * 80)

# Uncomment below if you want to manually enter token
# login(token="hf_your_token_here")

In [ ]:
print(f"Loading base model: {REWARD_BASE_MODEL}...")
print("Initializing model components\n")

# Load tokenizer
rm_tokenizer = AutoTokenizer.from_pretrained(REWARD_BASE_MODEL, trust_remote_code=True)
if rm_tokenizer.pad_token is None:
    rm_tokenizer.pad_token = rm_tokenizer.eos_token

# Load base model with appropriate device placement for DataParallel
# DataParallel requires all parameters on cuda:0, so we use explicit device placement
num_gpus = torch.cuda.device_count()
model_kwargs = {
    "torch_dtype": torch.bfloat16,
    "trust_remote_code": True,
    "low_cpu_mem_usage": True,
}

# For multi-GPU with DataParallel, load to cuda:0 explicitly
# For single GPU, use auto placement
if num_gpus > 1:
    model_kwargs["device_map"] = {"": 0}  # Load everything to cuda:0 for DataParallel
    print(f"✓ Loading model to cuda:0 (for DataParallel across {num_gpus} GPUs)")
else:
    model_kwargs["device_map"] = "auto"
    print("✓ Loading model with auto device placement")


# Flash Attention
if USE_FLASH_ATTENTION:
    model_kwargs["attn_implementation"] = ATTN_IMPLEMENTATION
    print("✓ Flash Attention 2 enabled")

base_model = AutoModelForCausalLM.from_pretrained(
    REWARD_BASE_MODEL,
    **model_kwargs
)

# Gradient checkpointing for memory efficiency
base_model.gradient_checkpointing_enable()
print("✓ Gradient checkpointing enabled (memory efficient)")

# Create reward model
reward_model = RewardModel(
    base_model=base_model,
    hidden_dim=RM_HIDDEN_DIM,
    head_type=RM_HEAD_TYPE,
    dropout=RM_DROPOUT
)

# Move reward head to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reward_model.reward_head = reward_model.reward_head.to(device).to(torch.bfloat16)

# Fix for "None of the inputs have requires_grad=True" warning
reward_model.base_model.enable_input_require_grads()
print("✓ Input gradients enabled (fixes gradient chain from embeddings)")

# ===========================
# GRADIENT FIX - PROPER UNFREEZING
# ===========================
print("\n" + "="*60)
print("🔧 APPLYING GRADIENT FIX")
print("="*60)

# 1. Freeze everything first
for param in reward_model.parameters():
    param.requires_grad = False

# 2. Unfreeze the last N layers of the base model
# Gemma-2-2b has 26 layers (0 to 25)
total_layers = 26
layers_to_unfreeze = range(total_layers - RM_UNFROZEN_LAYERS, total_layers)

print(f"Unfreezing layers: {list(layers_to_unfreeze)}")

for name, param in reward_model.named_parameters():
    # Unfreeze specific transformer layers
    if any(f"layers.{i}." in name for i in layers_to_unfreeze):
        param.requires_grad = True
    # ALWAYS unfreeze the reward head
    if "reward_head" in name:
        param.requires_grad = True

# 3. Double check (Crucial for fixing the Warning)
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in reward_model.parameters())

print(f"✅ Gradient Fix Applied!")
print(f"   Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.1f}%)")
print(f"   Total parameters: {total_params:,}")
print(f"   Frozen parameters: {total_params - trainable_params:,}")
print("="*60)

# Use single GPU for better memory management
# DataParallel adds significant overhead with gradient synchronization

print(f"\n✓ Using single GPU (cuda:0) for optimal memory efficiency")

print(f"  Note: Multi-GPU disabled to avoid DataParallel overhead")print(f"✓ Reward model created with {RM_HEAD_TYPE} head")

## Create DataLoaders

Initialize PyTorch DataLoaders for training and validation datasets.

In [ ]:
# Create datasets
train_dataset = PreferenceDataset(train_data, rm_tokenizer, max_length=RM_MAX_LENGTH)
val_dataset = PreferenceDataset(val_data, rm_tokenizer, max_length=RM_MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=RM_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=RM_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Check GPU device configuration
print("GPU Device Configuration:")
print(f"Number of GPUs available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
    # Show model device
    print("\nModel placement:")
    device_params = sum(p.numel() for p in reward_model.parameters() if 'cuda' in str(p.device))
    total_params = sum(p.numel() for p in reward_model.parameters())
    print(f"✓ Parameters on GPU: {device_params/1e6:.2f}M ({100*device_params/total_params:.1f}%)")
else:
    print("⚠️  Running on CPU")

## Training Setup

Configure optimizer, learning rate scheduler, and loss function for reward model training.

In [ ]:
# Progressive margin schedule: start easy, increase difficulty
def get_margin_for_epoch(epoch, total_epochs):
    """Progressive margin: 0.01 -> 0.05 over training (adjusted for actual data)"""
    return 0.01 + (0.04 * epoch / total_epochs)

# Bradley-Terry loss with margin for pairwise preferences
def bradley_terry_loss(chosen_rewards, rejected_rewards, margin=0.5):
    """
    Bradley-Terry model loss with margin: -log(sigmoid(r_chosen - r_rejected - margin))
    
    The model learns to make r_chosen > r_rejected by at least 'margin'.
    This encourages stronger preference differentiation.
    """
    # Compute difference with margin requirement
    diff = chosen_rewards - rejected_rewards - margin
    
    # Add numerical stability with clipping
    diff = torch.clamp(diff, min=-10, max=10)
    
    # Loss: -log(sigmoid(diff))
    loss = -torch.log(torch.sigmoid(diff) + 1e-8)
    return loss.mean()

# Get trainable parameters
trainable_params = [p for p in reward_model.parameters() if p.requires_grad]

# Optimizer with weight decay for regularization
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=RM_LEARNING_RATE,
    weight_decay=RM_WEIGHT_DECAY
)

# Learning rate scheduler (with longer warmup for stability)
num_training_steps = len(train_loader) * RM_EPOCHS // RM_GRADIENT_ACCUMULATION_STEPS
lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=num_training_steps // 5,  # 20% warmup for better stability
    num_training_steps=num_training_steps
)

# Initialize wandb (optional)
if USE_WANDB:
    import wandb
    wandb.init(
        project=WANDB_PROJECT,
        name="reward-model-training",
        config={
            'learning_rate': RM_LEARNING_RATE,
            'batch_size': RM_BATCH_SIZE,
            'epochs': RM_EPOCHS,
            'base_model': REWARD_BASE_MODEL,
            'head_type': RM_HEAD_TYPE,
        }
    )
    print("✓ W&B initialized")
else:
    print("✓ W&B disabled")

print("\nTraining setup complete!")
print(f"Total training steps: {num_training_steps}")
print(f"\nTraining on English and French preferences jointly")
print(f"  Train: {len(train_data)} pairs")
print(f"  Val: {len(val_data)} pairs")

## Training Loop

Define training and validation functions with progress monitoring and metric tracking.

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, gradient_accumulation_steps=1, margin=0.5):
    model.train()
    total_loss = 0
    total_accuracy = 0
    total_margin = 0
    num_batches = 0
    grad_norms = []
    
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f"Training (margin={margin:.2f})")
    for step, batch in enumerate(pbar):
        # Move to device
        chosen_input_ids = batch['chosen_input_ids'].to(device)
        chosen_attention_mask = batch['chosen_attention_mask'].to(device)
        rejected_input_ids = batch['rejected_input_ids'].to(device)
        rejected_attention_mask = batch['rejected_attention_mask'].to(device)
        
        # Forward pass
        chosen_rewards = model(chosen_input_ids, chosen_attention_mask)
        rejected_rewards = model(rejected_input_ids, rejected_attention_mask)
        
        # Calculate margin and accuracy for monitoring
        with torch.no_grad():
            margin_val = (chosen_rewards - rejected_rewards).mean().item()
            accuracy = (chosen_rewards > rejected_rewards).float().mean().item()
        
        # DEBUG: Check reward values (first batch only)
        if step == 0:
            print(f"\n[DEBUG] Batch 0 - First 5 samples:")
            print(f"  Chosen rewards: {chosen_rewards[:5].float().detach().cpu().numpy()}")
            print(f"  Rejected rewards: {rejected_rewards[:5].float().detach().cpu().numpy()}")
            print(f"  Difference: {(chosen_rewards - rejected_rewards)[:5].float().detach().cpu().numpy()}")
            print(f"  Target margin: {margin:.2f}")
            print(f"  Actual margin (mean): {margin_val:.4f}")
            print(f"  Accuracy: {accuracy:.2f}\n")
        
        # Compute loss with progressive margin
        loss = bradley_terry_loss(chosen_rewards, rejected_rewards, margin=margin)
        loss = loss / gradient_accumulation_steps
        
        # Backward pass
        loss.backward()
        
        total_loss += loss.item() * gradient_accumulation_steps
        total_accuracy += accuracy
        total_margin += margin_val
        num_batches += 1
        
        # Update weights
        if (step + 1) % gradient_accumulation_steps == 0:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        # Update progress bar with margin and accuracy
        pbar.set_postfix({
            'loss': f"{total_loss / num_batches:.4f}",
            'margin': f"{total_margin / num_batches:.4f}",
            'acc': f"{total_accuracy / num_batches:.2f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    # Print gradient statistics
    if grad_norms:
        print(f"  Gradient norm: mean={np.mean(grad_norms):.4f}, max={np.max(grad_norms):.4f}")
    
    # Print final epoch statistics
    avg_margin = total_margin / num_batches
    avg_accuracy = total_accuracy / num_batches
    print(f"  Reward Margin (chosen - rejected): {avg_margin:.4f}")
    print(f"  Accuracy (chosen > rejected): {avg_accuracy:.2f}")
    
    return total_loss / num_batches, total_accuracy / num_batches


def validate(model, loader, device, margin=0.5):
    model.eval()
    total_loss = 0
    total_accuracy = 0
    total_margin = 0
    num_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation"):
            chosen_input_ids = batch['chosen_input_ids'].to(device)
            chosen_attention_mask = batch['chosen_attention_mask'].to(device)
            rejected_input_ids = batch['rejected_input_ids'].to(device)
            rejected_attention_mask = batch['rejected_attention_mask'].to(device)
            
            chosen_rewards = model(chosen_input_ids, chosen_attention_mask)
            rejected_rewards = model(rejected_input_ids, rejected_attention_mask)
            
            # Calculate margin for monitoring
            margin_val = (chosen_rewards - rejected_rewards).mean().item()
            
            # Use same margin as training for consistency
            loss = bradley_terry_loss(chosen_rewards, rejected_rewards, margin=margin)
            accuracy = (chosen_rewards > rejected_rewards).float().mean()
            
            total_loss += loss.item()
            total_accuracy += accuracy.item()
            total_margin += margin_val
            num_batches += 1
    
    avg_margin = total_margin / num_batches
    print(f"  Validation Reward Margin: {avg_margin:.4f}")
    
    return total_loss / num_batches, total_accuracy / num_batches

print("Training functions defined (with margin and accuracy monitoring)")


In [ ]:
# Define device for sanity check
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Quick sanity check - BEFORE training (random initialization expected)
print("=" * 80)
print("SANITY CHECK - Pre-Training (Random Initialization Expected)")
print("=" * 80)

reward_model.eval()
with torch.no_grad():
    # Test on first batch
    sample_batch = next(iter(train_loader))
    chosen_ids = sample_batch['chosen_input_ids'][:1].to(device)
    chosen_mask = sample_batch['chosen_attention_mask'][:1].to(device)
    rejected_ids = sample_batch['rejected_input_ids'][:1].to(device)
    rejected_mask = sample_batch['rejected_attention_mask'][:1].to(device)
    
    r_chosen = reward_model(chosen_ids, chosen_mask).item()
    r_rejected = reward_model(rejected_ids, rejected_mask).item()
    
    print(f"\nReward Model Output (Untrained - Random Init):")
    print(f"  Chosen reward: {r_chosen:.4f}")
    print(f"  Rejected reward: {r_rejected:.4f}")
    print(f"  Difference: {r_chosen - r_rejected:.4f}")
    print(f"  Currently correct? {r_chosen > r_rejected}")
    
    print(f"\n⚠️  NOTE: Rewards are similar because model is randomly initialized.")
    print(f"           Training will learn to differentiate chosen vs rejected.")
    
    # Check data
    print(f"\nData Validation:")
    print(f"  Margin (expected difference): {sample_batch['margin'][0]:.4f}")
    print(f"  Data is ready for training ✓")
    
print("=" * 80)

In [ ]:
# Check for label consistency in preference data
print("\n" + "=" * 80)
print("DATA QUALITY CHECK - Preference Label Consistency")
print("=" * 80)

inconsistent_count = 0
for i, item in enumerate(preference_data[:1000]):  # Check first 1000 samples
    # Check if chosen_score > rejected_score (should be true for valid preferences)
    if item['chosen_score'] <= item['rejected_score']:
        inconsistent_count += 1
        if inconsistent_count <= 5:  # Show first 5 examples
            print(f"\n⚠️  Sample {i}: INCONSISTENT LABEL")
            print(f"   Chosen score:   {item['chosen_score']:.4f}")
            print(f"   Rejected score: {item['rejected_score']:.4f}")
            print(f"   Margin:         {item['margin']:.4f} (should be > 0)")

print(f"\nSample consistency check (first 1000 samples):")
print(f"  Inconsistent labels: {inconsistent_count}/1000 ({100*inconsistent_count/1000:.1f}%)")

if inconsistent_count > 100:
    print(f"\n🔴 CRITICAL: {inconsistent_count/10:.1f}% of data has inverted labels!")
    print(f"   The chosen translations have LOWER scores than rejected ones.")
    print(f"   This will prevent the reward model from learning properly.")
    print(f"\n   Fix: Re-run notebook 1 (synthetic_data_generation.ipynb)")
    print(f"   Make sure preference generation correctly identifies better translations.")
elif inconsistent_count > 0:
    print(f"\n🟡 WARNING: {100*inconsistent_count/1000:.1f}% of samples have inverted labels")
    print(f"   This is acceptable but may impact convergence.")
else:
    print(f"\n✓ All labels are consistent (chosen > rejected)")

print("=" * 80)


In [ ]:
# GPU memory monitoring
def print_gpu_memory():
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    else:
        print("No GPU available")

print("GPU Memory Monitor defined")
print_gpu_memory()

## Layer Freezing and Unfreezing Configuration

Configure which model parameters are trainable:
1. Freeze entire model initially
2. Unfreeze last N layers (specified by RM_UNFROZEN_LAYERS)
3. Always unfreeze the reward head (custom classification layer)

This ensures proper gradient flow and prevents optimization warnings.

In [ ]:
# --- APPLY PARAMETER FREEZING/UNFREEZING ---
print("\n" + "="*80)
print("🔥 CONFIGURING TRAINABLE PARAMETERS")
print("="*80)

# 1. Freeze the entire model first
for param in reward_model.parameters():
    param.requires_grad = False

print("✅ Step 1: Froze all parameters")

# 2. Unfreeze only the last N layers of the base model
# Gemma-2-2b has 26 layers
total_layers = 26
layers_to_train = range(total_layers - RM_UNFROZEN_LAYERS, total_layers)

print(f"✅ Step 2: Unfreezing layers {list(layers_to_train)} (last {RM_UNFROZEN_LAYERS} layers)")

for name, param in reward_model.named_parameters():
    # Unfreeze specific transformer layers
    if any(f"layers.{i}." in name for i in layers_to_train):
        param.requires_grad = True
    # ALWAYS unfreeze the reward head
    if "reward_head" in name:
        param.requires_grad = True

# 3. Verify
total_params = sum(p.numel() for p in reward_model.parameters())
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\n✅ Step 3: Parameter Verification Complete!")
print(f"  Total Parameters:     {total_params:,}")
print(f"  Trainable Parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"  Frozen Parameters:    {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")

# Display which components are trainable
print(f"\n📊 Trainable Components:")
trainable_components = set()
for name, param in reward_model.named_parameters():
    if param.requires_grad:
        component = name.split('.')[0]
        trainable_components.add(component)

for component in sorted(trainable_components):
    component_params = sum(p.numel() for n, p in reward_model.named_parameters() 
                          if p.requires_grad and n.startswith(component))
    print(f"  • {component}: {component_params:,} parameters")

print("="*80)

if trainable_params == 0:
    print("⚠️  WARNING: No trainable parameters! Training will fail!")
else:
    print(f"✅ Ready to train {trainable_params:,} parameters!")

In [ ]:
# Train the model
import time
from datetime import timedelta

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\n" + "="*80)
print("🚀 INITIALIZING TRAINING")
print("="*80)
print(f"  Device: {device}")
print(f"  Epochs: {RM_EPOCHS}")
print(f"  Batch Size: {RM_BATCH_SIZE} (effective: {RM_BATCH_SIZE * RM_GRADIENT_ACCUMULATION_STEPS})")
print(f"  Learning Rate: {RM_LEARNING_RATE:.2e}")
print(f"\n  Initial GPU Memory:")
print(f"  ", end="")
print_gpu_memory()
print("="*80)

best_val_accuracy = 0
training_start_time = time.time()

for epoch in range(RM_EPOCHS):
    epoch_start_time = time.time()
    
    # Progressive margin: start easy (0.1), increase to hard (0.5)
    current_margin = get_margin_for_epoch(epoch, RM_EPOCHS)
    
    print("\n" + "="*80)
    print(f"🔄 EPOCH {epoch + 1}/{RM_EPOCHS} | Progressive Margin: {current_margin:.2f}")
    print("="*80)
    
    # Train with progressive margin
    train_loss, train_acc = train_epoch(
        reward_model,
        train_loader,
        optimizer,
        lr_scheduler,
        device,
        gradient_accumulation_steps=RM_GRADIENT_ACCUMULATION_STEPS,
        margin=current_margin
    )
    
    # Validate with same margin
    val_loss, val_acc = validate(reward_model, val_loader, device, margin=current_margin)
    
    # Calculate timing
    epoch_elapsed = time.time() - epoch_start_time
    total_elapsed = time.time() - training_start_time
    epochs_completed = epoch + 1
    epochs_remaining = RM_EPOCHS - epochs_completed
    estimated_remaining = (total_elapsed / epochs_completed) * epochs_remaining
    
    # Display results with better formatting
    print("\n" + "-"*80)
    print("📊 TRAINING METRICS")
    print("-"*80)
    print(f"  📈 Train Loss: {train_loss:.4f}  |  ✅ Train Accuracy: {train_acc*100:.2f}%")
    print(f"  📉 Val Loss:   {val_loss:.4f}  |  {'✅' if val_acc > 0.5 else '⚠️ '} Val Accuracy:   {val_acc*100:.2f}%")
    
    print("\n" + "-"*80)
    print("⏱️  TIMING INFORMATION")
    print("-"*80)
    print(f"  This Epoch:  {timedelta(seconds=int(epoch_elapsed))}")
    print(f"  Total Time:  {timedelta(seconds=int(total_elapsed))}")
    if epochs_remaining > 0:
        print(f"  Remaining:   ~{timedelta(seconds=int(estimated_remaining))}")
    
    print("\n" + "-"*80)
    print("💾 GPU MEMORY STATUS")
    print("-"*80)
    print("  ", end="")
    # GPU memory status
    print_gpu_memory()
    
    # Log to wandb
    if USE_WANDB:
        wandb.log({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_accuracy': train_acc,
            'val_loss': val_loss,
            'val_accuracy': val_acc
        })
    
    # Save best model
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        print(f"\n✓ New best validation accuracy: {best_val_accuracy:.4f}")
        print(f"Saving model to {REWARD_MODEL_COLD_START}...")
        
        # Save model
        REWARD_MODEL_COLD_START.mkdir(exist_ok=True, parents=True)
        
        torch.save({
            'model_state_dict': reward_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'val_accuracy': val_acc,
            'config': {
                'base_model': REWARD_BASE_MODEL,
                'head_type': RM_HEAD_TYPE,
                'hidden_dim': RM_HIDDEN_DIM
            }
        }, REWARD_MODEL_COLD_START / "reward_model.pt")
        
        # Save tokenizer
        rm_tokenizer.save_pretrained(REWARD_MODEL_COLD_START)
        print("✓ Model saved successfully")

# Final Summary
print("\n\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)

# Final training time
total_training_time = time.time() - training_start_time
avg_time_per_epoch = total_training_time / RM_EPOCHS

print(f"\n📊 FINAL RESULTS")
print("-"*80)
print(f"  🏆 Best Val Accuracy:  {best_val_accuracy*100:.2f}%")
print(f"  ⏱️  Total Time:         {timedelta(seconds=int(total_training_time))}")
print(f"  📈 Avg Time/Epoch:     {timedelta(seconds=int(avg_time_per_epoch))}")
print(f"  🌍 Languages:          English → Arabic, French → Arabic")
print(f"  💾 Model saved:        {REWARD_MODEL_COLD_START.name}/")
print("="*80)

if USE_WANDB:
    wandb.finish()

## Test Reward Model

Evaluate the trained reward model on sample translations from the validation set.

In [ ]:
# Test the trained reward model
reward_model.eval()

print("Testing reward model on sample translations...\n")
print("=" * 80)

# Get some test examples
test_samples = random.sample(val_data, min(5, len(val_data)))

for i, sample in enumerate(test_samples, 1):
    print(f"\nExample {i}:")
    print(f"Source: {sample['source'][:100]}...")
    
    # Prepare inputs
    chosen_text = f"Source: {sample['source']}\nTranslation: {sample['chosen']}"
    rejected_text = f"Source: {sample['source']}\nTranslation: {sample['rejected']}"
    
    chosen_tokens = rm_tokenizer(
        chosen_text,
        max_length=RM_MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    rejected_tokens = rm_tokenizer(
        rejected_text,
        max_length=RM_MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    # Get rewards
    with torch.no_grad():
        chosen_reward = reward_model(
            chosen_tokens['input_ids'],
            chosen_tokens['attention_mask']
        ).item()
        
        rejected_reward = reward_model(
            rejected_tokens['input_ids'],
            rejected_tokens['attention_mask']
        ).item()
    
    print(f"\nChosen translation: {sample['chosen'][:100]}...")
    print(f"Chosen reward: {chosen_reward:.4f} (original score: {sample['chosen_score']:.4f})")
    
    print(f"\nRejected translation: {sample['rejected'][:100]}...")
    print(f"Rejected reward: {rejected_reward:.4f} (original score: {sample['rejected_score']:.4f})")
    
    print(f"\nReward margin: {chosen_reward - rejected_reward:.4f}")
    print(f"Correct preference: {'✓' if chosen_reward > rejected_reward else '✗'}")
    print("=" * 80)